# Example Workflow

My usual workflow is to play around with necessary functions in a notebook and slowly combine them to a class. When this class is done, I copy it and all the import statements in a module. And then I can use is from analysis scripts, in other classes and in notebooks. This is a raw attempt at showing you how I'd approach the "project" I've given you.

## Base Class

In this notebook, I'll define the Base_project class as defined in the Project. 

In [8]:
#in the first cell I'll usually put all the import statements. When I notice somewhere 30 cells in that I need to import another class, I'll also put it in here:

#these I import by default, I'll use them anyway
import numpy as np;
import pandas as pd;
import os;


#all these are added sometime later
import re

In [45]:
# usually I'll define some skeleton for the class here and then add functions, when they work as expected. 
#I won't do that now, because it would make it very hard to recover the workflow, so here is just the skeleton:
class Base_project:
    
    project = 'Example_project';
    
    #you need one anyway
    def __init__(self):
        pass
    
    #list the available subjects and make them part of the object
    def list_subjects(self):
        pass
    
    #check for a given subject ID if there is data 
    def subject_exists(self,sub):
        pass
    
    #and for specific data type
    def data_exists(self,sub,data):
        pass

In [3]:
#first of all it would be a good idea to make the path to the data a part of the class:

class Base_project:
    
    project = 'Sugar_study';
    path = 'C:/Users/neugebauer/Documents/Python_workshop/toy_data/';

In [4]:
b = Base_project();

Now we can simulate the whole 'self' -structure by passing the skeleton object in functions.

First step is to use the information about the path to list the files and folders in a directory.
A quick google search tells us that the package `os` should be helpful here. Let's have a look at a function:

In [5]:
os.listdir??

Signature: os.listdir(path=None)
Docstring:
Return a list containing the names of the files in the directory.

path can be specified as either str or bytes.  If path is bytes,
  the filenames returned will also be bytes; in all other circumstances
  the filenames returned will be str.
If path is None, uses the path='.'.
On some platforms, path may also be specified as an open file descriptor;\
  the file descriptor must refer to a directory.
  If this functionality is unavailable, using it raises NotImplementedError.

The list is in arbitrary order.  It does not include the special
entries '.' and '..' even if they are present in the directory.
Type:      builtin_function_or_method


That sounds good. Let's try it on the directory:

In [6]:
files = os.listdir( b.path );
print(files)

['mysterious_string.pic', 's01', 's02', 's03', 's04', 's05', 's06', 's07', 's09', 's10', 's11', 's12', 's13', 's14', 's15']


That looks good. Except, there is also the string. So we need a way to only select the real ones. Here are two ways in increasing order of sophistication:

In [7]:
#check wether it starts with s
sub_files = [f for f in files if f.startswith('s')];
print(sub_files)

['s01', 's02', 's03', 's04', 's05', 's06', 's07', 's09', 's10', 's11', 's12', 's13', 's14', 's15']


That works, but imagine, we have a file called 'some_garbage.txt', that would also match. So the better way is to use regular expressions. How do we do regular expression in Python?
Google tells us to use the module `re`. Okay! (add it to the cell in th beginning.

The way this seems to work is to create a regex-object that then can match, search, etc. using re.compile(). Let's look at that method:

In [9]:
re.compile??

Signature: re.compile(pattern, flags=0)
Source:   
def compile(pattern, flags=0):
    "Compile a regular expression pattern, returning a Pattern object."
    return _compile(pattern, flags)
File:      c:\users\neugebauer\anaconda3\lib\re.py
Type:      function


Okay, cool. So it only wants the pattern, we can do that! The structure of all the folders we want is 's' followed by two digits. In regular expression, this is `s[0-9]{2}'

In [10]:
#compile the thing:
my_regex = re.compile( 's[0-9]{2}');

If you use tab completion, you see that there are a lot of different methods. You could play around or google which of these you need.

In [12]:
my_regex.fullmatch??

Signature: my_regex.fullmatch(string, pos=0, endpos=9223372036854775807)
Docstring: Matches against all of the string.
Type:      builtin_function_or_method


Playing around can mean you create a string that should match and one that shouldn't and see how you get the result you want:

In [16]:
should_match = 's92';
should_not_match = '43h';

print(my_regex.fullmatch(should_match))
print(my_regex.fullmatch(should_not_match))


<re.Match object; span=(0, 3), match='s92'>
None


hm... that looks weird. What about this:

In [17]:
print( my_regex.search(should_match))
print( my_regex.search(should_not_match))

<re.Match object; span=(0, 3), match='s92'>
None


Okay, so it returns None if it doesn't match and something otherwise. Maybe we can get a bool from that:

In [19]:
print( bool(my_regex.search(should_match) ) );
print( bool(my_regex.search(should_not_match) ) );

True
False


Now that's what I'm talking about. We got our list, we want to select only the entries, where this returns True. An if-condition does this for us:

In [20]:
sub_list = [sub for sub in files if my_regex.search( sub )];
print(sub_list)

['s01', 's02', 's03', 's04', 's05', 's06', 's07', 's09', 's10', 's11', 's12', 's13', 's14', 's15']


Nice! So this is all we need to list the subjects. Let's put it into a function:

In [21]:
def list_subjects(project):
    path = project.path;
    my_regex = re.compile( 's[0-9]{2}' );
    full_list = os.listdir(path);
    sub_list = [sub for sub in full_list if my_regex.search(sub) ];
    project.subjects = sub_list

In [22]:
list_subjects(b);
b.subjects

['s01',
 's02',
 's03',
 's04',
 's05',
 's06',
 's07',
 's09',
 's10',
 's11',
 's12',
 's13',
 's14',
 's15']

Of course, we can make the function a bit nicer:

In [24]:
def list_subjects(project):
    
    regex = re.compile( 's[0-9]{2}' );
    project.subjects = [ sub for sub in os.listdir( project.path ) if regex.search( sub ) ];
    
b = Base_project();
list_subjects(b);
b.subjects

['s01',
 's02',
 's03',
 's04',
 's05',
 's06',
 's07',
 's09',
 's10',
 's11',
 's12',
 's13',
 's14',
 's15']

Works like a charm. If we now want to put it into the class, we just replace 'project' with 'self' and that should work. We can also call the method in the `__init__`-method so this knowledge is available from the beginning:

In [34]:
class Base_project:
    
    project = 'Example_project';
    path = 'C:/Users/neugebauer/Documents/Python_workshop/toy_data/';
    
    #you need one anyway
    def __init__(self):
        self.list_subjects();
    
    #list the available subjects and make them part of the object
    def list_subjects(self):
        regex = re.compile( 's[0-9]{2}' );
        self.subjects = [ sub for sub in os.listdir( project.path ) if regex.search( sub ) ];

    #check for a given subject ID if there is data 
    def data_exists(self,sub):
        pass    
    

It would be really nice to have a convenience function in the class to check for a specific subID using the knowledge from list_subjects.

In [26]:
#first we need to make the correct string out of the integer we get:
subID = 1;
subID_str = 's{:02d}'.format(subID);
print(subID_str)

s01


and check wether it's in there:

In [27]:
subID_str in b.subjects

True

In [28]:
def subject_exists(project, subID):
    return 's{:02d}'.format(subID) in project.subjects

In [29]:
data_exists(b,1)

True

Also good. Now put it into the class:

In [47]:
class Base_project:
    
    project = 'Example_project';
    path = 'C:/Users/neugebauer/Documents/Python_workshop/toy_data/';
    
    #you need one anyway
    def __init__(self):
        self.list_subjects();
    
    #list the available subjects and make them part of the object
    def list_subjects(self):
        regex = re.compile( 's[0-9]{2}' );
        self.subjects = [ sub for sub in os.listdir( self.path ) if regex.search( sub ) ];

    #check for a given subject ID if there is data 
    def subject_exists(self,subID):
        return 's{:02d}'.format(subID) in self.subjects  
    

In [48]:
b = Base_project();
print(b.subjects)

['s01', 's02', 's03', 's04', 's05', 's06', 's07', 's09', 's10', 's11', 's12', 's13', 's14', 's15']


In [49]:
print( b.subject_exists(1) );
print( b.subject_exists(99));

True
False


There are three different data types in the folders. It would be nice to check for these specific. In this case there is all data for all subjects, but assume you have e.g. rating and SCR data and the Biopac let's you down. Then this would be cool. So you add a flag for data type. The three types here are: `food_ratings.csv` , `rating_sequences.csv` or `sequence_RT.csv`. These are long names. As flags something shorter would be useful. Like `food`, `rate` and `RT` for example. If you want to map these onto each other, you can use a dictionary:

In [50]:
file_map = {'food': 'food_ratings.csv', 'rate': 'rating_sequence', 'RT': 'sequence_RT.csv'};


now in the function you can just get the right file by using the key:

In [51]:
file = file_map['food'];
print(file)

food_ratings.csv


Still, how do handle the subID in this? There is a nice function in `os.path` that joins different parts together to an absolute path on your operating system:

In [53]:
os.path.join( 'C:\\','Users','Lukas','test.py')

'C:\\Users\\Lukas\\test.py'

In [55]:
s_str = 's{:02d}'.format(2);
print(s_str)

s02


In [57]:
filetype = 'food';
subID = 3;

path = b.path;

filepath =  os.path.join( path, 's{:02d}'.format(subID), file_map[filetype]);
print(filepath)

C:/Users/neugebauer/Documents/Python_workshop/toy_data/s03\food_ratings.csv


That looks nice. The weird mix of / and \ still works. Next step, find out if there is such a file. Luckily, os.path doesn't let us down:

In [58]:
os.path.exists( filepath )

True

Cool! Last step, put it into a function:

In [59]:
def data_exists(project, subID, filetype):
    file_map = {'food': 'food_ratings.csv', 'rate': 'rating_sequence', 'RT': 'sequence_RT.csv'};
    path = os.path.join( project.path, 's{:02d}'.format(subID), file_map[filetype]);
    return os.path.exists(path)
    

In [61]:
data_exists(b,2,'food')

True

This even throws an error if you type in the wrong data type:

In [62]:
data_exists(b,2,'not_a_key')

KeyError: 'not_a_key'

But you can be nice to yourself and add a more informative message:

In [63]:
def data_exists(project, subID, filetype):
    
    file_map = {'food': 'food_ratings.csv', 'rate': 'rating_sequence', 'RT': 'sequence_RT.csv'};
    if not filetype in file_map.keys():
        raise KeyError( "filetype has to be one of the following: ['food','rate','RT']")
    
    path = os.path.join( project.path, 's{:02d}'.format(subID), file_map[filetype]);
    return os.path.exists(path)
    

In [64]:
data_exists(b,2,'not_a_key')

KeyError: "filetype has to be one of the following: ['food','rate','RT']"

Again, put it in the class. Make file_map a class attribute:

In [70]:
class Base_project:
    
    project = 'Example_project';
    path = 'C:/Users/neugebauer/Documents/Python_workshop/toy_data/';
    file_map = {'food': 'food_ratings.csv', 'rate': 'rating_sequence', 'RT': 'sequence_RT.csv'};

    #you need one anyway
    def __init__(self):
        self.list_subjects();
    
    #list the available subjects and make them part of the object
    def list_subjects(self):
        regex = re.compile( 's[0-9]{2}' );
        self.subjects = [ sub for sub in os.listdir( self.path ) if regex.search( sub ) ];

    #check for a given subject ID if there is data 
    def subject_exists(self,subID):
        return 's{:02d}'.format(subID) in self.subjects  
    
    def data_exists( self, subID, filetype):
        if not filetype in self.file_map.keys():
            raise KeyError( "filetype has to be one of the following: ['food','rate','RT']")
        path = os.path.join( self.path, 's{:02d}'.format(subID), self.file_map[filetype]);
        return os.path.exists(path)


In [71]:
b = Base_project()

In [72]:
b.subject_exists(3)

True

In [73]:
b.data_exists(3,'food')

True

Following the same logic, we can add a function that returns the path to a specific file. This we can also use in `data_exists` to save some code:

In [76]:
class Base_project:
    
    project = 'Example_project';
    path = 'C:/Users/neugebauer/Documents/Python_workshop/toy_data/';
    file_map = {'food': 'food_ratings.csv', 'rate': 'rating_sequence', 'RT': 'sequence_RT.csv'};

    #you need one anyway
    def __init__(self):
        self.list_subjects();
    
    #list the available subjects and make them part of the object
    def list_subjects(self):
        regex = re.compile( 's[0-9]{2}' );
        self.subjects = [ sub for sub in os.listdir( self.path ) if regex.search( sub ) ];

    #check for a given subject ID if there is data 
    def subject_exists(self,subID):
        return 's{:02d}'.format(subID) in self.subjects  
    
    def data_exists(*args, **kwargs): # you could also just ask for subID and filetype. if there are many input arguments, this is easier.
        path = self.path_to_data(*args, **kwargs);
        return os.path.exists(path)

    def path_to_data(self, subID, filetype):
        if not filetype in self.file_map.keys():
            raise KeyError( "filetype has to be one of the following: ['food','rate','RT']")
        path = os.path.join( self.path, 's{:02d}'.format(subID), self.file_map[filetype]);
        return path

That's it. Put it into a file and you're done here.